# Assignment 1 - LLM Evaluation: Product Description Generation

**Course:** AI Performance Engineering  
**Due Date:** April 5, 2026

This notebook contains the complete solution for Assignment 1, covering:
1. Rubric definition
2. Description generation
3. Manual evaluation
4. Improvement cycle
5. Judge model creation
6. Judge analysis and comparison

---
## Task 1: Define Your Rubric (15 points)

Before generating or evaluating anything, we need a clear, repeatable scoring framework.

### 1.1 Criterion Definitions

For each criterion, we define explicit standards for **good**, **ok**, and **bad** ratings. These are stored in a structured dictionary for programmatic evaluation.

### 1.2 Criterion Thresholds Dictionary

In [1]:
# Criterion thresholds for automated evaluation
CRITERION_THRESHOLDS = {
    "fluency": {
        "criteria_type": "quality",
        "check": "manual",
        "good": {
            "description": "Natural, smooth sentences with varied structure. Easy to read aloud. No awkward phrasing or repetition.",
        },
        "ok": {
            "description": "Mostly natural but with minor awkwardness (e.g., one slightly repetitive phrase or choppy transition).",
        },
        "bad": {
            "description": "Multiple awkward phrases, unnatural word order, or repetitive structure that disrupts readability.",
        },
    },
    "grammar": {
        "criteria_type": "quality",
        "check": "manual",
        "good": {
            "description": "Zero spelling or punctuation errors. Proper sentence structure throughout.",
        },
        "ok": {
            "description": "One minor error (e.g., missing comma, minor typo) that doesn't affect comprehension.",
        },
        "bad": {
            "description": "Multiple errors or one major error (e.g., subject-verb disagreement, misspelled product name).",
        },
    },
    "tone": {
        "criteria_type": "quality",
        "check": "manual",
        "good": {
            "description": "Consistently friendly, credible sales voice. Enthusiastic without being pushy. Professional language appropriate for e-commerce.",
        },
        "ok": {
            "description": "Generally appropriate tone but with one instance of overly casual language, excessive hype, or slightly flat delivery.",
        },
        "bad": {
            "description": "Inappropriate tone (too formal/technical, too casual, or overly aggressive sales language). Multiple tone inconsistencies.",
        },
    },
    "length": {
        "criteria_type": "quality",
        "check": "automatic",
        "good": {
            "description": "50-90 words (inclusive)",
            "ranges": [(50, 90)],
        },
        "ok": {
            "description": "40-49 words OR 91-110 words",
            "ranges": [(40, 49), (91, 110)],
        },
        "bad": {
            "description": "Fewer than 40 words OR more than 110 words",
            "ranges": [(0, 39), (111, 999999)],
        },
    },
    "grounding": {
        "criteria_type": "quality",
        "check": "manual",
        "good": {
            "description": "All information comes directly from provided data (name, attributes, material, warranty). No fabricated features or specifications.",
        },
        "ok": {
            "description": "Minor embellishment that's reasonable inference (e.g., 'sleek design' when material is 'aluminum') but no false claims.",
        },
        "bad": {
            "description": "Contains fabricated information, incorrect specifications, or claims not supported by the provided data.",
        },
    },
    "latency": {
        "criteria_type": "objective",
        "check": "automatic",
        "good": {
            "description": "≤ 5000ms (5 seconds)",
            "ranges": [(0, 5000)],
        },
        "ok": {
            "description": "5001-10000ms (5-10 seconds)",
            "ranges": [(5001, 10000)],
        },
        "bad": {
            "description": "> 10000ms (10+ seconds)",
            "ranges": [(10001, 999999)],
        },
    },
    "cost": {
        "criteria_type": "objective",
        "check": "automatic",
        "good": {
            "description": "≤ $0.01 per description",
            "ranges": [(0, 0.01)],
        },
        "ok": {
            "description": "$0.011-$0.05 per description",
            "ranges": [(0.011, 0.05)],
        },
        "bad": {
            "description": "> $0.05 per description",
            "ranges": [(0.051, 999999)],
        },
    },
}

# split criteria into quality and objective
QUALITY_CRITERIA = [
    k for k, v in CRITERION_THRESHOLDS.items() if v.get("criteria_type") == "quality"
]
OBJECTIVE_CRITERIA = [
    k for k, v in CRITERION_THRESHOLDS.items() if v.get("criteria_type") == "objective"
]
EVALUATION_CRITERIA = list[str](CRITERION_THRESHOLDS.keys())


def evaluate_criterion(criterion_name: str, value: float) -> str:
    """
    Generic evaluation function for any criterion with numeric ranges.

    Args:
        criterion_name: Name of the criterion (e.g., 'length', 'latency', 'cost')
        value: Numeric value to evaluate

    Returns:
        'good', 'ok', or 'bad'
    """
    thresholds = CRITERION_THRESHOLDS[criterion_name]

    for rating in ["good", "ok", "bad"]:
        for min_val, max_val in thresholds[rating]["ranges"]:
            if min_val <= value <= max_val:
                return rating

    return "bad"


def evaluate_length(word_count: int) -> str:
    """Evaluate length criterion based on word count using CRITERION_THRESHOLDS."""
    return evaluate_criterion("length", word_count)


def evaluate_latency(latency_ms: float) -> str:
    """Evaluate latency criterion based on milliseconds using CRITERION_THRESHOLDS."""
    return evaluate_criterion("latency", latency_ms)


def evaluate_cost(cost_usd: float) -> str:
    """Evaluate cost criterion based on USD amount using CRITERION_THRESHOLDS."""
    return evaluate_criterion("cost", cost_usd)


def calculate_pass_fail(ratings: dict) -> str:
    """
    Calculate pass/fail based on ratings.

    Args:
        ratings: dict with keys ['fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost']
                 values: 'good', 'ok', or 'bad'

    Returns:
        'pass' or 'fail'

    Rules:
        - Automatic failure if grounding, grammar, or length is "bad"
        - Pass requires: ≥4 good, ≤1 bad, and ≥5 acceptable (good or ok)
    """
    # Go/no-go rules - automatic failure conditions
    if ratings["grounding"] == "bad":
        return "fail"
    if ratings["grammar"] == "bad":
        return "fail"
    if ratings["length"] == "bad":
        return "fail"

    # Cumulative pass bar - count ratings by type
    good_count = sum(1 for v in ratings.values() if v == "good")
    ok_count = sum(1 for v in ratings.values() if v == "ok")
    bad_count = sum(1 for v in ratings.values() if v == "bad")

    # Calculate combined acceptable ratings
    good_or_ok_count = good_count + ok_count

    # Must have: ≥4 good, ≤1 bad, and ≥5 acceptable (good or ok)
    if (good_count >= 4) and (bad_count <= 1) and (good_or_ok_count >= 5):
        return "pass"
    else:
        return "fail"

---
## Task 2: Generate Descriptions for Every Product (20 points)

Generate product descriptions using a language model from Nebius Token Factory.

In [41]:
# Import required libraries
import os
import time

import pandas as pd
from dotenv import load_dotenv
from openai import BadRequestError, OpenAI

load_dotenv()

# Constants
OUTPUT_FILE_PATH = "assignment_01.csv"
NEBIUS_API_BASE_URL = "https://api.tokenfactory.nebius.com/v1/"

# Initialize OpenAI client
client = OpenAI(base_url=NEBIUS_API_BASE_URL, api_key=os.environ.get("NEBIUS_API_KEY"))

print(EVALUATION_CRITERIA)

['fluency', 'grammar', 'tone', 'length', 'grounding', 'latency', 'cost']


In [3]:
# Load the product dataset
df = pd.read_csv("Assignment_01_product_dataset.csv")
print(f"Loaded {len(df)} products")
df.head(2)

Loaded 50 products


,product_name,Product_attribute_list,material,warranty
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S-Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1-year limited warranty


### 2.1 System Prompt

Design a prompt that instructs the model to generate persuasive 50-90 word product descriptions.

In [4]:
SYSTEM_PROMPT = """
You are an expert e-commerce copywriter. Your task is to write persuasive product descriptions for online shoppers.

Requirements:
- Length: Exactly 50-90 words
- Tone: Friendly, credible, and enthusiastic (but not pushy)
- Content: Use ONLY the provided product information - do not fabricate features
- Style: Natural, easy-to-read sentences with varied structure
- Grammar: Perfect spelling and punctuation

Focus on benefits and appeal to the target customer. Make them want to buy!

OUTPUT: Provide only the product description text. Do not include any preamble, explanation, or additional commentary.
""".strip()


def create_user_prompt(
    product_name: str, attributes: str, material: str, warranty: str
) -> str:
    return f"""Product Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

Write a persuasive product description (50-90 words)."""

### 2.2 Model Selection

Choose one model from Nebius Token Factory:
- Gemma-2-9b-it
- Meta-Llama-3.1-8B-Instruct

In [5]:
# Choosing the model
MODELS = {
    "gemma-2-9b-it": "google/gemma-2-9b-it-fast",
    "meta-llama-3.1-8b-instruct": "meta-llama/Meta-Llama-3.1-8B-Instruct",
}
MODEL_NAME = MODELS["meta-llama-3.1-8b-instruct"]


def generate_description(
    product_name: str,
    attributes: str,
    material: str,
    warranty: str,
    *,
    model_name: str = None,
    system_prompt: str = None,
    temperature: float = None,
    max_completion_tokens: int = None,
) -> dict:
    """
    Generate a product description and collect metrics.

    Args:
        product_name: Product name
        attributes: Product attributes
        material: Product material
        warranty: Product warranty
        model_name: Model to use (defaults to MODEL_NAME)
        system_prompt: System prompt (defaults to SYSTEM_PROMPT) (optional)
        temperature: Sampling temperature (optional)
        max_completion_tokens: Max tokens to generate

    Returns:
        dict with keys: generated_description, latency_ms, input_tokens, output_tokens
    """
    user_prompt = create_user_prompt(product_name, attributes, material, warranty)

    start_time = time.time()

    # Build request with optional parameters
    request_kwargs = {
        "model": model_name if model_name is not None else MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    }

    if max_completion_tokens is not None:
        request_kwargs["max_completion_tokens"] = max_completion_tokens

    if temperature is not None:
        request_kwargs["temperature"] = temperature

    response = client.chat.completions.create(**request_kwargs)

    end_time = time.time()
    latency_ms = int((end_time - start_time) * 1000)

    return {
        "generated_description": response.choices[0].message.content.strip(),
        "latency_ms": latency_ms,
        "input_tokens": response.usage.prompt_tokens,
        "output_tokens": response.usage.completion_tokens,
    }

### 2.3 Generate Descriptions for All Products

In [6]:
def generate_descriptions_for_df(
    source_df: pd.DataFrame,
    *,
    n_rows: int = None,
    model_name: str = None,
    system_prompt: str = None,
    temperature: float = None,
    max_completion_tokens: int = None,
    sleep_s: float = 0.5,
) -> pd.DataFrame:
    """
    Generate descriptions for multiple products in a DataFrame.

    Args:
        source_df: DataFrame with product data (must have columns:
                   product_name, Product_attribute_list, material, warranty)
        n_rows: Number of rows to process (None = all rows)
        model_name: Model to use (defaults to MODEL_NAME)
        system_prompt: System prompt (defaults to SYSTEM_PROMPT)
        temperature: Sampling temperature (optional)
        max_completion_tokens: Max tokens to generate
        sleep_s: Sleep duration between requests to avoid rate limiting

    Returns:
        DataFrame with original columns plus generated_description, latency_ms,
        input_tokens, output_tokens
    """
    subset = source_df.head(n_rows).copy() if n_rows else source_df.copy()
    results = []

    for idx, row in subset.iterrows():
        print(f"Processing {idx + 1}/{len(subset)}: {row['product_name']}")

        out = generate_description(
            product_name=row["product_name"],
            attributes=row["Product_attribute_list"],
            material=row["material"],
            warranty=row["warranty"],
            model_name=model_name,
            system_prompt=system_prompt,
            temperature=temperature,
            max_completion_tokens=max_completion_tokens,
        )

        results.append({**row.to_dict(), **out})
        time.sleep(sleep_s)

    print("\nGeneration complete!")
    return pd.DataFrame(results)

In [7]:
if not os.path.exists(OUTPUT_FILE_PATH):
    print("File does not exist, running generation")
    # Run generation for all products using the reusable helper
    results_df = generate_descriptions_for_df(
        df, n_rows=None, system_prompt=SYSTEM_PROMPT
    )

    # Add blank columns for evaluation criteria
    for criterion in EVALUATION_CRITERIA:
        results_df[criterion] = ""

    results_df["final_score"] = ""

    # Save
    results_df.to_csv(OUTPUT_FILE_PATH, index=False)
    print(f"Saved results to {OUTPUT_FILE_PATH}")
else:
    print("File exists, loading results")
    # Same shape as the generation branch (list of row dicts)
    results_df = pd.read_csv(OUTPUT_FILE_PATH)

# Display summary
print(f"\nGenerated {len(results_df)} descriptions")
print(f"Average latency: {results_df['latency_ms'].mean():.0f}ms")
print(f"Average input tokens: {results_df['input_tokens'].mean():.0f}")
print(f"Average output tokens: {results_df['output_tokens'].mean():.0f}")

results_df.head(1)

File does not exist, running generation
Processing 1/50: Apple iPhone 15 Pro
Processing 2/50: Samsung Galaxy S24 Ultra
Processing 3/50: Google Pixel 8 Pro
Processing 4/50: Sony WH-1000XM5 Headphones
Processing 5/50: Bose QuietComfort Ultra Earbuds
Processing 6/50: Amazon Echo Dot (5th Gen)
Processing 7/50: Dell XPS 13 9310 Laptop
Processing 8/50: Apple MacBook Air 13″ (M3)
Processing 9/50: Microsoft Surface Pro 10
Processing 10/50: Garmin Forerunner 255
Processing 11/50: Fitbit Charge 6
Processing 12/50: GoPro HERO12 Black
Processing 13/50: DJI Mini 4 Pro Drone
Processing 14/50: Nintendo Switch OLED
Processing 15/50: PlayStation 5 Slim
Processing 16/50: Xbox Series X
Processing 17/50: Instant Pot Duo 6-Quart
Processing 18/50: Keurig K-Supreme Plus Smart
Processing 19/50: Vitamix 5200 Blender
Processing 20/50: Dyson V15 Detect Vacuum
Processing 21/50: iRobot Roomba j7+
Processing 22/50: Yeti Rambler 20 oz Tumbler
Processing 23/50: Stanley Quencher H2.0 40 oz
Processing 24/50: Hydro Flas

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency,cost,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,"""Get ready to experience the ultimate smartpho...",3881,200,111,,,,,,,,


---
## Task 3: Manual (Human) Evaluation (10 points)

Manually evaluate 10-15 products using the rubric defined in Task 1.

### 3.1 Add Cost Column

Calculate the cost per description based on token usage and model pricing.

In [8]:
# Fetch pricing dynamically from Nebius API
def get_model_pricing(model_name: str) -> tuple[float, float]:
    """
    Fetch pricing for a specific model from Nebius Token Factory API.

    Args:
        model_name: The model identifier (e.g., "meta-llama/Meta-Llama-3.1-8B-Instruct")

    Returns:
        Tuple of (input_price_per_1k_tokens, output_price_per_1k_tokens) in USD
    """
    try:
        # List all models with verbose=true to get pricing information
        # Note: The OpenAI client doesn't support query params directly,
        # so we need to make a raw HTTP request
        import requests

        response = requests.get(
            f"{NEBIUS_API_BASE_URL}models?verbose=true",
            headers={"Authorization": f"Bearer {os.environ.get('NEBIUS_API_KEY')}"},
        )
        response.raise_for_status()
        models_data = response.json()

        # Find the specific model
        for model in models_data.get("data", []):
            if model.get("id") == model_name:
                pricing = model.get("pricing", {})

                # Pricing values are strings representing price per token
                # Convert to float and then to per 1K tokens
                input_price_per_token = float(pricing.get("prompt", "0"))
                output_price_per_token = float(pricing.get("completion", "0"))

                input_price_per_1k = input_price_per_token * 1000
                output_price_per_1k = output_price_per_token * 1000

                print(f"✓ Fetched pricing for {model_name}:")
                print(f"  Input:  ${input_price_per_1k:.6f} per 1K tokens")
                print(f"  Output: ${output_price_per_1k:.6f} per 1K tokens")
                return input_price_per_1k, output_price_per_1k

        # Model not found
        raise ValueError(f"Model not found: {model_name}")

    except Exception as e:
        print(f"⚠ Warning: Could not fetch pricing from API: {e}")
        print("Using fallback pricing values for meta-llama/Meta-Llama-3.1-8B-Instruct")
        # Fallback: $0.02/1M input, $0.06/1M output = $0.00002/1K, $0.00006/1K
        # NOTE: This is the pricing for the model used in the assignment ("meta-llama/Meta-Llama-3.1-8B-Instruct")
        return (0.00002, 0.00006)


# Get pricing for the chosen model
PRICE_PER_1K_INPUT_TOKENS, PRICE_PER_1K_OUTPUT_TOKENS = get_model_pricing(MODEL_NAME)

# Calculate cost (numeric; overwrites any placeholder in the cost column)
results_df["cost_value"] = (
    results_df["input_tokens"] / 1000 * PRICE_PER_1K_INPUT_TOKENS
) + (results_df["output_tokens"] / 1000 * PRICE_PER_1K_OUTPUT_TOKENS)

print(f"\nAverage cost per description: ${results_df['cost_value'].mean():.6f}")
print(
    f"Total cost for {len(results_df)} descriptions: ${results_df['cost_value'].sum():.6f}"
)

results_df.to_csv(OUTPUT_FILE_PATH, index=False, na_rep="")

✓ Fetched pricing for meta-llama/Meta-Llama-3.1-8B-Instruct:
  Input:  $0.000020 per 1K tokens
  Output: $0.000060 per 1K tokens

Average cost per description: $0.000010
Total cost for 50 descriptions: $0.000510


### 3.2 Manual Evaluation Instructions

**Manually evaluate 10-15 products**

1. Open OUTPUT_FILE_PATH (`assignment_01`)
2. Select 10-15 diverse products
3. For each selected product, rate each criterion (fluency, grammar, tone, length, grounding, latency, cost) as:
   - `good`
   - `ok`
   - `bad`
4. Use the rubric definitions from Task 1

After completing manual evaluation, run the cell below to load and analyze your scores.

In [9]:
def evaluate_automatic_criteria(df: pd.DataFrame, n_rows: int = None) -> pd.DataFrame:
    """
    Evaluate automatic criteria (length, latency, cost).

    Args:
        df: The dataframe to evaluate
        n_rows: Number of rows to evaluate (None = all rows)

    Returns:
        DataFrame with automatic evaluations
    """
    # Read
    # Empty cells are often inferred as float64; rubric values are strings.
    for col in ("length", "latency"):
        if col in df.columns:
            df[col] = df[col].astype(object)

    # Limit to first n rows if specified
    rows_to_eval = df.head(n_rows) if n_rows else df

    # Evaluate length (count words in generated_description)
    word_counts = rows_to_eval["generated_description"].apply(
        lambda x: len(str(x).split())
    )
    # evaluate_length / evaluate_latency / evaluate_cost each take one numeric arg
    # and call evaluate_criterion internally (same rubric as evaluate_criterion("…", x))
    df.loc[rows_to_eval.index, "word_counts"] = word_counts
    df.loc[rows_to_eval.index, "length"] = word_counts.map(evaluate_length)

    df.loc[rows_to_eval.index, "latency"] = rows_to_eval["latency_ms"].map(
        evaluate_latency
    )

    df.loc[rows_to_eval.index, "cost"] = df.loc[rows_to_eval.index, "cost_value"].apply(
        lambda x: evaluate_cost(x) if isinstance(x, (int, float)) else x
    )

    return df


# Evaluate first 15 rows and save
automatic_eval_df = evaluate_automatic_criteria(results_df, n_rows=10)[
    [
        "product_name",
        "Product_attribute_list",
        "material",
        "warranty",
        "generated_description",
        "latency_ms",
        "input_tokens",
        "output_tokens",
        "word_counts",
        "cost_value",
        *EVALUATION_CRITERIA,
        "final_score",
    ]
]

print("\nRows after automatic evaluations:")
automatic_eval_df.head(1)


Rows after automatic evaluations:


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,word_counts,cost_value,fluency,grammar,tone,length,grounding,latency,cost,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,"""Get ready to experience the ultimate smartpho...",3881,200,111,88.0,0.000011,,,,good,,good,good,


In [10]:
# # One entry per product_name in automatic_eval_df (edit defaults / overrides after review).
# _MANUAL_RATING_DEFAULTS: dict[str, str] = {
#     "fluency": "ok",
#     "grammar": "ok",
#     "tone": "ok",
#     "grounding": "ok",
# }
# MANUAL_RATINGS: dict[str, dict[str, str]] = {
#     name: {**_MANUAL_RATING_DEFAULTS}
#     for name in automatic_eval_df["product_name"].unique()
# }

In [11]:
MANUAL_RATINGS_TEMP = {
    "Apple iPhone 15 Pro": {
        "fluency": "ok",  # Awkward phrasing
        "grammar": "good",
        "tone": "ok",  # "Unleash the power", "boasts"
        "grounding": "ok",  # superlatives
    },
    "Samsung Galaxy S24 Ultra": {
        "fluency": "ok",  # Awkward phrasing
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",  # MCW?
    },
    "Google Pixel 8 Pro": {
        "fluency": "ok",  # "exudes"
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",  # superlatives
    },
    "Sony WH-1000XM5 Headphones": {
        "fluency": "good",
        "grammar": "good",
        "tone": "ok",
        "grounding": "ok",
    },
    "Bose QuietComfort Ultra Earbuds": {
        "fluency": "good",
        "grammar": "good",
        "tone": "ok",  # "Step into a world of pure tranquility"
        "grounding": "ok",
    },
    "Amazon Echo Dot (5th Gen)": {
        "fluency": "ok",  # unnatural "lifelike"
        "grammar": "good",
        "tone": "ok",
        "grounding": "ok",
    },
    "Dell XPS 13 9310 Laptop": {
        "fluency": "ok",
        "grammar": "ok",
        "tone": "ok",
        "grounding": "ok",
    },
    "Apple MacBook Air 13″ (M3)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Microsoft Surface Pro 10": {
        "fluency": "ok",
        "grammar": "ok",
        "tone": "ok",
        "grounding": "ok",
    },
    "Garmin Forerunner 255": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",
    },
}


def merge_manual_ratings(df: pd.DataFrame, manual_ratings: dict) -> pd.DataFrame:
    # Merge the manual_ratings dict into automatic_eval_df
    for product_name, ratings in manual_ratings.items():
        df.loc[df["product_name"] == product_name, "fluency"] = ratings["fluency"]
        df.loc[df["product_name"] == product_name, "grammar"] = ratings["grammar"]
        df.loc[df["product_name"] == product_name, "tone"] = ratings["tone"]
        df.loc[df["product_name"] == product_name, "grounding"] = ratings["grounding"]

    return df


automatic_eval_df = merge_manual_ratings(automatic_eval_df, MANUAL_RATINGS_TEMP)

### 3.3 Final Score

**Calculate `final_score` (pass/fail) using the formula from Task 1**

In [12]:
def calculate_final_score(df: pd.DataFrame) -> pd.DataFrame:
    criteria = [c for c in EVALUATION_CRITERIA if c in df.columns]

    if not criteria:
        print("No criteria to evaluate")
        return df

    def _row_has_all_criteria(row: pd.Series) -> bool:
        for col in criteria:
            val = row[col]

            if pd.isna(val):
                return False

            if isinstance(val, str) and val.strip() == "":
                return False

        return True

    complete_mask = df.apply(_row_has_all_criteria, axis=1)

    for idx in df.index[complete_mask]:
        ratings = {c: df.at[idx, c] for c in criteria}
        df.at[idx, "final_score"] = calculate_pass_fail(ratings)

    return df


final_score_df = calculate_final_score(automatic_eval_df)

In [13]:
# save final_score_df
final_score_df.to_csv(OUTPUT_FILE_PATH, index=False, na_rep="")
final_score_df.head(1)

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,word_counts,cost_value,fluency,grammar,tone,length,grounding,latency,cost,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,"""Get ready to experience the ultimate smartpho...",3881,200,111,88.0,0.000011,ok,good,ok,good,ok,good,good,pass


### 3.4 Baseline Analysis

**Document your findings**

Based on the manual evaluation:

1. **Best performing criteria:**
   - `length` - Maybe becasue it's the simplest to enforce (rigid rule), AND I added the `max_completion_tokens=150` param with the call.
   - `cost` - Maybe I set too-loose restrictions, but also, this is a simpler model and task so it's reasonable every run is cheap.
   - In terms of quality checks, `grammar` is almost perfect. This is probably since LLMs are trained on massive amount of text, specifically in English, so they provide no typos or nonsense.

2. **Worst performing criteria:**
   - `grounding` - The outputs include exaggerated adjectives (e.g., "escape to serenity" "unlock the ultimate productivity experience!")

3. **Strategy for improvement (Task 4):**
   - To improve the `grounding` issues, we can expand the system prompt to include something like "Use a simple tone - don't add any superlatives. Also, stick ONLY to the provided info and don't add anything in your generated description.", and we can also play with the temperature if applicable (lower == less creative).

---
## Task 4: Improvement Cycle (15 points)

Iterate to achieve better results based on Task 3 baseline analysis.

### Experiment Template

For each experiment, document:
1. **What you changed**
2. **Why you expected it to help**
3. **New evaluation scores**

Keep code for successful experiments. Document failed experiments but code is optional.

In [14]:
def apply_task23_metrics(
    df: pd.DataFrame,
    *,
    n_rows_for_auto_eval: int = None,
    clear_manual_criteria: bool = False,
    price_per_1k_input: float = None,
    price_per_1k_output: float = None,
) -> pd.DataFrame:
    """
    Apply Task 2/3 metrics to a DataFrame with generated descriptions.

    This function:
    1. Computes cost_value from token counts
    2. Evaluates automatic criteria (length, latency, cost)
    3. Optionally clears manual criteria columns for re-evaluation

    Args:
        df: DataFrame with generated descriptions and token counts
        n_rows_for_auto_eval: Number of rows to evaluate (None = all rows)
        clear_manual_criteria: If True, clear manual criteria columns
                               (fluency, grammar, tone, grounding, final_score)
        price_per_1k_input: Price per 1K input tokens (defaults to PRICE_PER_1K_INPUT_TOKENS)
        price_per_1k_output: Price per 1K output tokens (defaults to PRICE_PER_1K_OUTPUT_TOKENS)

    Returns:
        DataFrame with cost_value and automatic evaluations added
    """
    df = df.copy()

    # Use provided pricing or fall back to global variables
    input_price = (
        price_per_1k_input
        if price_per_1k_input is not None
        else PRICE_PER_1K_INPUT_TOKENS
    )
    output_price = (
        price_per_1k_output
        if price_per_1k_output is not None
        else PRICE_PER_1K_OUTPUT_TOKENS
    )

    # Compute cost
    df["cost_value"] = (
        df["input_tokens"] / 1000 * input_price
        + df["output_tokens"] / 1000 * output_price
    )

    # Apply automatic criteria evaluation
    df = evaluate_automatic_criteria(df, n_rows=n_rows_for_auto_eval)

    # Optionally clear manual criteria for re-rating
    if clear_manual_criteria:
        manual_cols = ["fluency", "grammar", "tone", "grounding", "final_score"]
        rows_idx = (
            df.head(n_rows_for_auto_eval).index if n_rows_for_auto_eval else df.index
        )
        for col in manual_cols:
            if col in df.columns:
                df.loc[rows_idx, col] = ""

    return df

### Experiment 1: Prompt engineering

**What changed:**
- Rewrite the system prompt to enforce stricter constraints

**Why expected to help:**
- `grounding` received not-good-enough results due to exaggerated adjectives in the generated description. By adding "Use a simple tone - don't add any superlatives. Also, stick ONLY to the provided info and don't add anything in your generated description" - we expect more relaxed outputs.

**Results:**
- MUCH BETTER descriptions. Overall improvement for almost all relevant outputs. This new, more restrictive and specific system prompt helped.

In [16]:
# Experiment 1: Stricter prompt for better grounding

EXP1_SYSTEM_PROMPT = """
You are an expert e-commerce copywriter. Your task is to write product descriptions for online shoppers.

CRITICAL REQUIREMENTS:
- Length: Exactly 50-90 words
- Tone: Simple, clear, and factual - NO superlatives or hype words
- Content: Use ONLY the provided product information - do NOT fabricate, infer, or exaggerate any features
- Style: Natural, easy-to-read sentences
- Grammar: Perfect spelling and punctuation

Avoid words like: ultimate, best, revolutionary, amazing, incredible, perfect, etc.
Stick to the facts provided. Make it appealing through clarity, not exaggeration.

OUTPUT: Provide only the product description text. Do not include any preamble or explanation.
""".strip()

# Configuration
EXP1_N_ROWS = 10  # Same as baseline for fair comparison
# EXP1_TEMPERATURE = None  # Lower temperature for more deterministic output
EXP1_MAX_COMPLETION_TOKENS = 150
EXP1_OUTPUT_PATH = "assignment_01_task4_exp1.csv"

if not os.path.exists(EXP1_OUTPUT_PATH):
    # Generate descriptions with new prompt
    exp1_df = generate_descriptions_for_df(
        df,
        n_rows=EXP1_N_ROWS,
        max_completion_tokens=EXP1_MAX_COMPLETION_TOKENS,
        system_prompt=EXP1_SYSTEM_PROMPT,
    )

    # Add blank columns for evaluation criteria
    for criterion in EVALUATION_CRITERIA:
        exp1_df[criterion] = ""

    exp1_df["final_score"] = ""

    # Apply metrics
    exp1_df = apply_task23_metrics(
        exp1_df,
        n_rows_for_auto_eval=EXP1_N_ROWS,
        clear_manual_criteria=True,  # Clear for manual re-evaluation
    )
else:
    exp1_df = pd.read_csv(EXP1_OUTPUT_PATH)

Processing 1/10: Apple iPhone 15 Pro
Processing 2/10: Samsung Galaxy S24 Ultra
Processing 3/10: Google Pixel 8 Pro
Processing 4/10: Sony WH-1000XM5 Headphones
Processing 5/10: Bose QuietComfort Ultra Earbuds
Processing 6/10: Amazon Echo Dot (5th Gen)
Processing 7/10: Dell XPS 13 9310 Laptop
Processing 8/10: Apple MacBook Air 13″ (M3)
Processing 9/10: Microsoft Surface Pro 10
Processing 10/10: Garmin Forerunner 255

Generation complete!


In [18]:
exp1_df.head(2)

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,cost_value,word_counts,length,latency,cost,fluency,grammar,tone,grounding,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,The Apple iPhone 15 Pro is designed for perfor...,2862,223,104,0.000011,82,good,good,good,good,good,good,good,pass
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S-Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1-year limited warranty,The Samsung Galaxy S24 Ultra features a 200 MP...,2064,226,107,0.000011,80,good,good,good,good,good,good,bad,fail


In [19]:
MANUAL_RATINGS_EXP1 = {
    "Apple iPhone 15 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Samsung Galaxy S24 Ultra": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "bad",  #  `Its 6.8" 120 Hz AMOLED display` - no 6.8" mentioned
    },
    "Google Pixel 8 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Sony WH-1000XM5 Headphones": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Bose QuietComfort Ultra Earbuds": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",
    },
    "Amazon Echo Dot (5th Gen)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Dell XPS 13 9310 Laptop": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Apple MacBook Air 13″ (M3)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Microsoft Surface Pro 10": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Garmin Forerunner 255": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
}

exp1_df = merge_manual_ratings(exp1_df, MANUAL_RATINGS_EXP1)
exp1_df = calculate_final_score(exp1_df)
exp1_df.to_csv(EXP1_OUTPUT_PATH, index=False, na_rep="")

In [20]:
exp1_df.head(2)

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,cost_value,word_counts,length,latency,cost,fluency,grammar,tone,grounding,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,The Apple iPhone 15 Pro is designed for perfor...,2862,223,104,0.000011,82,good,good,good,good,good,good,good,pass
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S-Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1-year limited warranty,The Samsung Galaxy S24 Ultra features a 200 MP...,2064,226,107,0.000011,80,good,good,good,good,good,good,bad,fail


### Experiment 2: Decoding parameters

**What changed:**
- Lowered temperature (lower == less creative).

**Why expected to help:**
- The LLM won't add inforamtion not from the original dataset.

**Results:**
- Best yet (among original run + 1st experiment above) - all reviews got a passing score :)

In [22]:
# Experiment 2: Lower temperature only (keep original prompt)

# Configuration
EXP2_N_ROWS = 10
EXP2_TEMPERATURE = 0.2  # Very low temperature for minimal creativity
EXP2_OUTPUT_PATH = "assignment_01_task4_exp2.csv"

if not os.path.exists(EXP2_OUTPUT_PATH):
    # Generate descriptions with lower temperature
    exp2_df = generate_descriptions_for_df(
        df,
        n_rows=EXP2_N_ROWS,
        system_prompt=SYSTEM_PROMPT,  # Use original prompt
        temperature=EXP2_TEMPERATURE,
    )

    # Apply metrics
    exp2_df = apply_task23_metrics(
        exp2_df,
        n_rows_for_auto_eval=EXP2_N_ROWS,
        clear_manual_criteria=True,
    )
else:
    exp2_df = pd.read_csv(EXP2_OUTPUT_PATH)

Processing 1/10: Apple iPhone 15 Pro
Processing 2/10: Samsung Galaxy S24 Ultra
Processing 3/10: Google Pixel 8 Pro
Processing 4/10: Sony WH-1000XM5 Headphones
Processing 5/10: Bose QuietComfort Ultra Earbuds
Processing 6/10: Amazon Echo Dot (5th Gen)
Processing 7/10: Dell XPS 13 9310 Laptop
Processing 8/10: Apple MacBook Air 13″ (M3)
Processing 9/10: Microsoft Surface Pro 10
Processing 10/10: Garmin Forerunner 255

Generation complete!


In [24]:
exp2_df.head(2)

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,cost_value,word_counts,length,latency,cost,fluency,grammar,tone,grounding,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,"""Experience the ultimate in smartphone power a...",2080,200,96,0.000010,75,good,good,good,good,good,good,good,pass
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S-Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1-year limited warranty,"""Capture life's moments with unparalleled clar...",3240,203,109,0.000011,84,good,good,good,good,good,good,ok,pass


In [25]:
MANUAL_RATINGS_EXP2 = {
    "Apple iPhone 15 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Samsung Galaxy S24 Ultra": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",
    },
    "Google Pixel 8 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Sony WH-1000XM5 Headphones": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Bose QuietComfort Ultra Earbuds": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "ok",  # `Advanced Noise Cancellation (ANC)` - it's supposed to be `Active Noise Cancellation (ANC)`
    },
    "Amazon Echo Dot (5th Gen)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Dell XPS 13 9310 Laptop": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Apple MacBook Air 13″ (M3)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Microsoft Surface Pro 10": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Garmin Forerunner 255": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
}

exp2_df = merge_manual_ratings(exp2_df, MANUAL_RATINGS_EXP2)
exp2_df = calculate_final_score(exp2_df)
exp2_df.to_csv(EXP2_OUTPUT_PATH, index=False, na_rep="")

### Experiment 3: Combined Approach

**What changed:**
- Combined stricter prompt (from Exp 1) with lower temperature (0.2, from Exp 2)

**Why expected to help:**
- The combination of explicit grounding instructions and reduced sampling randomness should maximize factual accuracy while maintaining natural language quality.

**Results:**
- ALMOST PERFECT. All processed items recieved the highest, `good`, result for each criterion except for a single `latency` value ("ok") which varies per run.

In [26]:
# Experiment 3: Combined approach (stricter prompt + lower temperature)

# Configuration
EXP3_N_ROWS = 10
EXP3_OUTPUT_PATH = "assignment_01_task4_exp3.csv"

if not os.path.exists(EXP3_OUTPUT_PATH):
    # Generate descriptions with both improvements
    exp3_df = generate_descriptions_for_df(
        df,
        n_rows=EXP3_N_ROWS,
        system_prompt=EXP1_SYSTEM_PROMPT,  # Use stricter prompt from Exp 1
        temperature=EXP2_TEMPERATURE,  # same as Exp 2
    )

    # Apply metrics
    exp3_df = apply_task23_metrics(
        exp3_df,
        n_rows_for_auto_eval=EXP3_N_ROWS,
        clear_manual_criteria=True,
    )
else:
    exp3_df = pd.read_csv(EXP3_OUTPUT_PATH)

Processing 1/10: Apple iPhone 15 Pro
Processing 2/10: Samsung Galaxy S24 Ultra
Processing 3/10: Google Pixel 8 Pro
Processing 4/10: Sony WH-1000XM5 Headphones
Processing 5/10: Bose QuietComfort Ultra Earbuds
Processing 6/10: Amazon Echo Dot (5th Gen)
Processing 7/10: Dell XPS 13 9310 Laptop
Processing 8/10: Apple MacBook Air 13″ (M3)
Processing 9/10: Microsoft Surface Pro 10
Processing 10/10: Garmin Forerunner 255

Generation complete!


In [28]:
exp3_df.head(2)

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,cost_value,word_counts,length,latency,cost,fluency,grammar,tone,grounding,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,The Apple iPhone 15 Pro features a powerful A1...,5777,223,85,0.000010,68,good,ok,good,good,good,good,good,pass
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S-Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1-year limited warranty,The Samsung Galaxy S24 Ultra is a high-perform...,3678,226,104,0.000011,81,good,good,good,good,good,good,good,pass


In [29]:
MANUAL_RATINGS_EXP3 = {
    "Apple iPhone 15 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Samsung Galaxy S24 Ultra": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Google Pixel 8 Pro": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Sony WH-1000XM5 Headphones": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Bose QuietComfort Ultra Earbuds": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Amazon Echo Dot (5th Gen)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Dell XPS 13 9310 Laptop": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Apple MacBook Air 13″ (M3)": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Microsoft Surface Pro 10": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
    "Garmin Forerunner 255": {
        "fluency": "good",
        "grammar": "good",
        "tone": "good",
        "grounding": "good",
    },
}

exp3_df = merge_manual_ratings(exp3_df, MANUAL_RATINGS_EXP3)
exp3_df = calculate_final_score(exp3_df)
exp3_df.to_csv(EXP3_OUTPUT_PATH, index=False, na_rep="")

---
## Task 5: Create a Judge Model (20 points)

Build an automated LLM judge that grades descriptions using the Task 1 rubric.

### 5.1 Judge Model Selection

Start with the model you **did not** use in Task 2. If it struggles, switch to a larger model.

In [30]:
# Choose judge model (the one NOT used in Task 2)
JUDGE_MODEL_NAME = MODELS["gemma-2-9b-it"]  # or another model if needed
# Prompt + completion must fit in this limit (see provider / error message when too large).
JUDGE_MODEL_CONTEXT_TOKENS = 8192

print(f'Judge model: "{JUDGE_MODEL_NAME}"\n(Generator model was: "{MODEL_NAME})"')

Judge model: "google/gemma-2-9b-it-fast"
(Generator model was: "meta-llama/Meta-Llama-3.1-8B-Instruct)"


### 5.2 Pydantic Schema for Structured Output

Define the output schema. Note: **explanation comes before verdict** (important for chain-of-thought reasoning).

In [31]:
from typing import Literal

from pydantic import BaseModel, Field


class CriterionEvaluation(BaseModel):
    explanation: str = Field(
        description="Brief reasoning (2-3 short sentences; no repetition; stay under ~400 characters)",
        max_length=450,
    )
    verdict: Literal["good", "ok", "bad"] = Field(
        description="Rating: good, ok, or bad"
    )


class DescriptionEvaluation(BaseModel):
    fluency: CriterionEvaluation
    grammar: CriterionEvaluation
    tone: CriterionEvaluation
    length: CriterionEvaluation
    grounding: CriterionEvaluation


# Display schema
print("Judge output schema:")
print(DescriptionEvaluation.model_json_schema())

Judge output schema:
{'$defs': {'CriterionEvaluation': {'properties': {'explanation': {'description': 'Brief reasoning (2-3 short sentences; no repetition; stay under ~400 characters)', 'maxLength': 450, 'title': 'Explanation', 'type': 'string'}, 'verdict': {'description': 'Rating: good, ok, or bad', 'enum': ['good', 'ok', 'bad'], 'title': 'Verdict', 'type': 'string'}}, 'required': ['explanation', 'verdict'], 'title': 'CriterionEvaluation', 'type': 'object'}}, 'properties': {'fluency': {'$ref': '#/$defs/CriterionEvaluation'}, 'grammar': {'$ref': '#/$defs/CriterionEvaluation'}, 'tone': {'$ref': '#/$defs/CriterionEvaluation'}, 'length': {'$ref': '#/$defs/CriterionEvaluation'}, 'grounding': {'$ref': '#/$defs/CriterionEvaluation'}}, 'required': ['fluency', 'grammar', 'tone', 'length', 'grounding'], 'title': 'DescriptionEvaluation', 'type': 'object'}


**Why explanation before verdict?**

`explanation` comes before `verdict` in the schema to encourage the LLM to provide rubric-grounded judgments and useful explanations for analysis.

### Why it helps for an LLM judge

- **Think-then-label:**  
  You want the model to ground the rating in the text first, then map that reasoning to good / ok / bad. Declaring explanation first (and often listing it first in the schema) nudges generation toward "reason → verdict" instead of "verdict → justify".

- **Less shallow rationalization:**  
  If verdict were first, some models tend to pick a label early and then write a short, generic explanation that matches it. Putting reasoning first makes that pattern a bit harder and often yields more specific, rubric-linked explanations.

> *Note: this is not a guarantee to provide better results—the model can still tweak its evaluation.*

### 5.3 Judge Prompt

Write a prompt that embeds the Task 1 rubric and provides necessary context for evaluation.

In [32]:
LLM_JUDGE_EVALUATION_CRITERIA_STRING = ""

for name in QUALITY_CRITERIA:
    LLM_JUDGE_EVALUATION_CRITERIA_STRING += f"{name.lower()}:\n"

    for threshold in ["good", "ok", "bad"]:
        LLM_JUDGE_EVALUATION_CRITERIA_STRING += f'\t- "{threshold}": {CRITERION_THRESHOLDS[name.lower()][threshold]["description"]}\n'

    LLM_JUDGE_EVALUATION_CRITERIA_STRING += "\n"

JUDGE_SYSTEM_PROMPT = f"""
You are an expert evaluator of product descriptions. Your task is to rate product descriptions according to specific criteria.

For each criterion, provide:
1. An explanation of your reasoning
2. A verdict: 'good', 'ok', or 'bad'

EVALUATION CRITERIA:
{LLM_JUDGE_EVALUATION_CRITERIA_STRING}
Be objective and consistent in your evaluations.
Keep each explanation concise (about 2-4 sentences). The reply must be one complete JSON object matching the schema - do not truncate.
""".strip()


def create_judge_prompt(
    description: str, product_name: str, attributes: str, material: str, warranty: str
) -> str:
    return f"""PRODUCT INFORMATION:
Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

GENERATED DESCRIPTION:
{description}

Evaluate this description according to the criteria (fluency, grammar, tone, length, grounding)."""

In [33]:
print(JUDGE_SYSTEM_PROMPT)

You are an expert evaluator of product descriptions. Your task is to rate product descriptions according to specific criteria.

For each criterion, provide:
1. An explanation of your reasoning
2. A verdict: 'good', 'ok', or 'bad'

EVALUATION CRITERIA:
fluency:
	- "good": Natural, smooth sentences with varied structure. Easy to read aloud. No awkward phrasing or repetition.
	- "ok": Mostly natural but with minor awkwardness (e.g., one slightly repetitive phrase or choppy transition).
	- "bad": Multiple awkward phrases, unnatural word order, or repetitive structure that disrupts readability.

grammar:
	- "good": Zero spelling or punctuation errors. Proper sentence structure throughout.
	- "ok": One minor error (e.g., missing comma, minor typo) that doesn't affect comprehension.
	- "bad": Multiple errors or one major error (e.g., subject-verb disagreement, misspelled product name).

tone:
	- "good": Consistently friendly, credible sales voice. Enthusiastic without being pushy. Professiona

### 5.4 Judge Implementation

In [34]:
def judge_description(
    description: str, product_name: str, attributes: str, material: str, warranty: str
) -> DescriptionEvaluation:
    """
    Use the judge model to evaluate a product description.

    Returns:
        DescriptionEvaluation object with ratings for each criterion
    """
    user_prompt = create_judge_prompt(
        description, product_name, attributes, material, warranty
    )

    # Initial completion budget (~4 chars/token heuristic). If we still truncate, retry using
    # usage.prompt_tokens from the first response to allocate the rest of the context window.
    char_len = len(JUDGE_SYSTEM_PROMPT) + len(user_prompt)
    approx_prompt_tokens = char_len // 4 + 256
    max_out = JUDGE_MODEL_CONTEXT_TOKENS - approx_prompt_tokens - 48
    if max_out < 512:
        raise ValueError(
            "Judge prompt too long for JUDGE_MODEL_CONTEXT_TOKENS; shorten inputs or pick a "
            "larger-context judge model and update JUDGE_MODEL_CONTEXT_TOKENS."
        )

    def _create(mt: int):
        return client.chat.completions.create(
            model=JUDGE_MODEL_NAME,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.3,  # Lower temperature for more consistent evaluation
            max_tokens=mt,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "description_evaluation",
                    "schema": DescriptionEvaluation.model_json_schema(),
                },
            },
        )

    try:
        response = _create(max_out)
    except BadRequestError:
        max_out = max(512, max_out - 1024)
        response = _create(max_out)

    choice = response.choices[0]
    usage = getattr(response, "usage", None)

    if (
        choice.finish_reason == "length"
        and usage is not None
        and usage.prompt_tokens is not None
    ):
        retry_max = JUDGE_MODEL_CONTEXT_TOKENS - int(usage.prompt_tokens) - 24
        if retry_max > max_out:
            try:
                response = _create(retry_max)
            except BadRequestError:
                response = _create(max(512, retry_max - 512))
            choice = response.choices[0]

    if choice.finish_reason == "length":
        raise RuntimeError(
            "Judge still truncated JSON after using the full context window for completion. "
            "Shorten JUDGE_SYSTEM_PROMPT / rubric text, tighten per-criterion explanations, or use "
            "a judge model with a larger context."
        )

    content = choice.message.content

    if not content or not str(content).strip():
        raise ValueError("Judge model returned empty content")

    text = str(content).strip()

    if text.startswith("```"):
        lines = text.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    return DescriptionEvaluation.model_validate_json(text)

---
## Task 6: Run and Analyze the Judge (20 points)

Run the judge model and compare its evaluations to human ratings.

### 6.1 Sanity Check (5 products)

In [35]:
# Run judge on 5 products for sanity check
results_df = pd.read_csv(OUTPUT_FILE_PATH)

# Select 5 products (can be random or specific)
sanity_check_indices = [0, 10, 20, 30, 40]  # Adjust as needed

print("Sanity Check - Judge Evaluations:")
print("=" * 80)

for idx in sanity_check_indices:
    row = results_df.iloc[idx]

    print(f"\nProduct: {row['product_name']}")
    print(f"Description: {row['generated_description']}")

    evaluation = judge_description(
        description=row["generated_description"],
        product_name=row["product_name"],
        attributes=row["Product_attribute_list"],
        material=row["material"],
        warranty=row["warranty"],
    )

    print("\nJudge Ratings:")
    for criterion in ["fluency", "grammar", "tone", "length", "grounding"]:
        eval_obj = getattr(evaluation, criterion)
        print(f"  {criterion}: {eval_obj.verdict}")
        print(f"    → {eval_obj.explanation}")

    time.sleep(1)  # Rate limiting

print("\n" + "=" * 80)

Sanity Check - Judge Evaluations:

Product: Apple iPhone 15 Pro
Description: Unleash the power of innovation with the Apple iPhone 15 Pro. Built to impress, this sleek device boasts a lightning-fast A17 Pro chip and stunning 120 Hz ProMotion display that responds to every touch. Wrapped in a durable titanium frame and protected by Ceramic Shield glass, it's the perfect blend of style and substance. Fast charging via USB-C gets you back on the go in no time. Get the iPhone 15 Pro and experience the future of smartphone technology, backed by a 1-year limited warranty.

Judge Ratings:
  fluency: ok
    → The sentences flow well and have varied structure. There's a slight overuse of adverbs like 'lightning-fast' and 'stunning', but it doesn't significantly disrupt readability.
  grammar: good
    → No spelling or punctuation errors. Sentence structure is correct throughout.
  tone: ok
    → The tone is enthusiastic and positive, using language appropriate for a sales pitch. However, phrase

**Sanity Check Analysis:**

The judge outputs above MOSTLY make sense, but are definitely UNRELIABLE AND SOMETIMES WRONG. It quotes words from the descriptions to support its verdict, which adds persuasion.

It does not apply our rubric correctly, with some prominent issues I found:
- `length` - it <u>consistently</u> misses the actual word count and indicates incorrect values.<br>
For example: `"The description is 98 words, which falls into the 'bad' category (more than 110 words)"` - the decription was actually **76 words** and in any case - it is less than 110 words so 'bad' is a big mistake to score.

- `grounding` - It found a problem in a certain description that mentioned "battery-life" for a product that has no battery. Therefore, it (the judge) should have scored that description as "bad", but it gave it "ok", although the `grounding` rule specifically mentions that if some decription `"contains fabricated information"` it should be scored as "bad".

### 6.2 Full Run - Judge All Products

In [36]:
# Run judge on all products
results_df = pd.read_csv(OUTPUT_FILE_PATH)

# Add judge columns
judge_columns = [
    "judge_fluency",
    "judge_fluency_explanation",
    "judge_grammar",
    "judge_grammar_explanation",
    "judge_tone",
    "judge_tone_explanation",
    "judge_length",
    "judge_length_explanation",
    "judge_grounding",
    "judge_grounding_explanation",
    "judge_final_score",
]

for col in judge_columns:
    if col not in results_df.columns:
        results_df[col] = ""

print(f"Running judge on {len(results_df)} products...")

for idx, row in results_df.iterrows():
    print(f"Judging {idx + 1}/{len(results_df)}: {row['product_name']}")

    evaluation = judge_description(
        description=row["generated_description"],
        product_name=row["product_name"],
        attributes=row["Product_attribute_list"],
        material=row["material"],
        warranty=row["warranty"],
    )

    # Store judge ratings
    for criterion in QUALITY_CRITERIA:
        eval_obj = getattr(evaluation, criterion)
        results_df.at[idx, f"judge_{criterion}"] = eval_obj.verdict
        results_df.at[idx, f"judge_{criterion}_explanation"] = eval_obj.explanation

    # Calculate judge final score using Task 1 formula
    judge_ratings = {
        "fluency": results_df.at[idx, "judge_fluency"],
        "grammar": results_df.at[idx, "judge_grammar"],
        "tone": results_df.at[idx, "judge_tone"],
        "length": results_df.at[idx, "judge_length"],
        "grounding": results_df.at[idx, "judge_grounding"],
        "latency": results_df.at[idx, "latency"],  # From Task 2
        "cost": results_df.at[idx, "cost"],  # From Task 3
    }

    # Apply pass/fail formula from Task 1
    results_df.at[idx, "judge_final_score"] = calculate_pass_fail(judge_ratings)

    time.sleep(1)  # Rate limiting

# Save updated results
results_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nJudge evaluation complete! Results saved to {OUTPUT_FILE_PATH}")
results_df.head(1)

Running judge on 50 products...
Judging 1/50: Apple iPhone 15 Pro
Judging 2/50: Samsung Galaxy S24 Ultra
Judging 3/50: Google Pixel 8 Pro
Judging 4/50: Sony WH-1000XM5 Headphones
Judging 5/50: Bose QuietComfort Ultra Earbuds
Judging 6/50: Amazon Echo Dot (5th Gen)
Judging 7/50: Dell XPS 13 9310 Laptop
Judging 8/50: Apple MacBook Air 13″ (M3)
Judging 9/50: Microsoft Surface Pro 10
Judging 10/50: Garmin Forerunner 255
Judging 11/50: Fitbit Charge 6
Judging 12/50: GoPro HERO12 Black
Judging 13/50: DJI Mini 4 Pro Drone
Judging 14/50: Nintendo Switch OLED
Judging 15/50: PlayStation 5 Slim
Judging 16/50: Xbox Series X
Judging 17/50: Instant Pot Duo 6-Quart
Judging 18/50: Keurig K-Supreme Plus Smart
Judging 19/50: Vitamix 5200 Blender
Judging 20/50: Dyson V15 Detect Vacuum
Judging 21/50: iRobot Roomba j7+
Judging 22/50: Yeti Rambler 20 oz Tumbler
Judging 23/50: Stanley Quencher H2.0 40 oz
Judging 24/50: Hydro Flask 32 oz Wide Mouth
Judging 25/50: Contigo Autoseal West Loop 16 oz
Judging 26/50

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,word_counts,cost_value,...,judge_fluency_explanation,judge_grammar,judge_grammar_explanation,judge_tone,judge_tone_explanation,judge_length,judge_length_explanation,judge_grounding,judge_grounding_explanation,judge_final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1-year limited warranty,Unleash the power of innovation with the Apple...,3719,200,106,85.0,0.00001,...,The sentences are well-structured and flow smo...,good,The description is grammatically correct with ...,good,The tone is enthusiastic and persuasive withou...,good,"The description is 88 words long, falling with...",good,All information presented is directly derived ...,pass


### 6.3 Compare Judge vs Human Evaluation

In [37]:
# Load results with both human and judge evaluations
results_df = pd.read_csv(OUTPUT_FILE_PATH)

# Filter rows with human evaluation
compared = results_df[
    (results_df["fluency"] != "") & (results_df["fluency"].notna())
].copy()

print(f"Comparing judge vs human on {len(compared)} products\n")
print("Agreement Rates by Criterion:")
print("=" * 50)

for criterion in QUALITY_CRITERIA:
    human_col = criterion
    judge_col = f"judge_{criterion}"

    if human_col in compared.columns and judge_col in compared.columns:
        agreements = (compared[human_col] == compared[judge_col]).sum()
        total = len(compared)
        agreement_rate = (agreements / total * 100) if total > 0 else 0

        print(f"\n{criterion.upper()}:")
        print(f"  Agreement: {agreements}/{total} ({agreement_rate:.1f}%)")

        # Show disagreements
        disagreements = compared[compared[human_col] != compared[judge_col]]
        if len(disagreements) > 0:
            print("  Disagreements:")
            for _, row in disagreements.iterrows():
                print(
                    f"    - {row['product_name'][:40]}: Human={row[human_col]}, Judge={row[judge_col]}"
                )


Comparing judge vs human on 10 products

Agreement Rates by Criterion:

FLUENCY:
  Agreement: 4/10 (40.0%)
  Disagreements:
    - Apple iPhone 15 Pro: Human=ok, Judge=good
    - Sony WH-1000XM5 Headphones: Human=good, Judge=ok
    - Bose QuietComfort Ultra Earbuds: Human=good, Judge=ok
    - Amazon Echo Dot (5th Gen): Human=ok, Judge=good
    - Dell XPS 13 9310 Laptop: Human=ok, Judge=good
    - Microsoft Surface Pro 10: Human=ok, Judge=good

GRAMMAR:
  Agreement: 6/10 (60.0%)
  Disagreements:
    - Samsung Galaxy S24 Ultra: Human=good, Judge=ok
    - Bose QuietComfort Ultra Earbuds: Human=good, Judge=ok
    - Dell XPS 13 9310 Laptop: Human=ok, Judge=good
    - Microsoft Surface Pro 10: Human=ok, Judge=good

TONE:
  Agreement: 5/10 (50.0%)
  Disagreements:
    - Apple iPhone 15 Pro: Human=ok, Judge=good
    - Sony WH-1000XM5 Headphones: Human=ok, Judge=good
    - Amazon Echo Dot (5th Gen): Human=ok, Judge=good
    - Dell XPS 13 9310 Laptop: Human=ok, Judge=good
    - Microsoft Surface

**Analysis of Judge vs Human Agreement:**

Across **10** manually rated products, exact label agreement is modest everywhere except a stark gap on **grounding**.<br>
**Grammar** aligns most often (**6/10, 60%**).<br>
**Tone** is **5/10 (50%)**; **length** is **4/10 (40%)**.<br>
**Fluency** is **4/10 (40%)**.<br>
**Grounding** is **1/10 (10%)**: the only exact match is **Apple MacBook Air 13″ (M3)** (both `good`). In **nine** cases the judge and human differ; the dominant pattern is Human=`ok` vs Judge=`good`, with one important exception—**Bose QuietComfort Ultra Earbuds**, where the judge scores **`bad`** (e.g. unsupported “Academy Award–winning design”) while the human stayed at **`ok`**.

1. **Where do they agree most?**
   - **Grammar (60%)** — highest agreement, but still only six matches. Remaining rows split between the **judge** tightening to **`ok`** where humans rated **`good`** (Samsung, Bose) and the judge more generous on polished copy (Human=`ok`, Judge=`good` on Dell XPS 13 and Microsoft Surface Pro 10).

2. **Where do they diverge?**
   - **Fluency (40%)** — six disagreements. Several are Human=`ok`, Judge=`good` (iPhone 15 Pro, Echo Dot, Dell XPS, Surface Pro), suggesting the judge reads marketing fluency as stronger than the human does; others flip the other way (Sony, Bose: Human=`good`, Judge=`ok`).
   - **Tone (50%)** — five disagreements; in each listed case the human chose **`ok`** and the judge **`good`**, so the model consistently rates the sales voice more positively.
   - **Length (50%)** — five disagreements; humans often mark descriptions **`good`** on length while the judge applies the word-count rubric more strictly and outputs **`ok`** (Pixel 8 Pro, Sony, Bose, Dell, Garmin).
   - **Grounding (10%)** — aside from MacBook, every row disagrees: mostly Human=`ok` vs Judge=`good`, plus the Bose case where the judge is **`bad`** and the human is still **`ok`**—showing both systematic optimism on the judge side and one sharp mismatch in severity.

3. **Why might these differences occur?**
   - **Rubric calibration** — humans may use “`ok` unless clearly wrong or excellent,” while the judge often upgrades fluent, on-brief copy to **`good`** (especially tone and grounding).
   - **Grounding vs verification** — without a single external fact sheet in the loop, the judge still sometimes invents confidence (“everything is supported”) whereas humans reserve **`good`** for clearly attribute-faithful text; conversely, the judge may flag a specific fabricated claim (Bose) that the human overlooked or graded more leniently.
   - **Subjective and banded criteria** — tone depends on unstated brand voice; length depends on how strictly word intervals are applied. Small differences in counting or thresholds (`good` vs `ok` bands) produce many `good`/`ok` splits.

### 6.4 Criterion-by-Criterion Judging

Run the judge separately for each criterion (one API call per criterion per product).

In [40]:
def _single_criterion_judge_system_prompt(criterion: str) -> str:
    """System prompt with rubric for one quality criterion only (isolated judging)."""
    c = criterion.lower()
    rubric_lines = [f"{c}:"]
    for threshold in ["good", "ok", "bad"]:
        rubric_lines.append(
            f'\t- "{threshold}": {CRITERION_THRESHOLDS[c][threshold]["description"]}'
        )
    rubric_block = "\n".join(rubric_lines)
    return f"""You are an expert evaluator of product descriptions.
Evaluate ONLY the {c} criterion. Do not discuss or rate any other dimension.
Provide:
1. An explanation of your reasoning (2-4 short sentences)
2. A verdict: 'good', 'ok', or 'bad'

EVALUATION CRITERION:
{rubric_block}

Be objective and consistent. Reply must be one complete JSON object matching the schema - do not truncate.
""".strip()


def judge_single_criterion(
    description: str,
    product_name: str,
    attributes: str,
    material: str,
    warranty: str,
    criterion: str,
) -> CriterionEvaluation:
    """
    Judge a single criterion in isolation.

    Args:
        criterion: One of 'fluency', 'grammar', 'tone', 'length', 'grounding'
    """
    c = criterion.lower()
    if c not in QUALITY_CRITERIA:
        raise ValueError(
            f"criterion must be one of {QUALITY_CRITERIA}, got {criterion!r}"
        )

    system_prompt = _single_criterion_judge_system_prompt(c)
    criterion_prompt = f"""PRODUCT INFORMATION:
Name: {product_name}
Attributes: {attributes}
Material: {material}
Warranty: {warranty}

GENERATED DESCRIPTION:
{description}

Evaluate ONLY the {c.upper()} of this description according to the rubric."""

    char_len = len(system_prompt) + len(criterion_prompt)
    approx_prompt_tokens = char_len // 4 + 256
    max_out = JUDGE_MODEL_CONTEXT_TOKENS - approx_prompt_tokens - 48

    if max_out < 256:
        raise ValueError(
            "Judge prompt too long for JUDGE_MODEL_CONTEXT_TOKENS; shorten inputs or pick a "
            "larger-context judge model and update JUDGE_MODEL_CONTEXT_TOKENS."
        )

    def _create(mt: int):
        return client.chat.completions.create(
            model=JUDGE_MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": criterion_prompt},
            ],
            temperature=0.3,
            max_tokens=mt,
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "criterion_evaluation",
                    "schema": CriterionEvaluation.model_json_schema(),
                },
            },
        )

    try:
        response = _create(max_out)
    except BadRequestError:
        max_out = max(256, max_out - 1024)
        response = _create(max_out)

    choice = response.choices[0]
    usage = getattr(response, "usage", None)

    if (
        choice.finish_reason == "length"
        and usage is not None
        and usage.prompt_tokens is not None
    ):
        retry_max = JUDGE_MODEL_CONTEXT_TOKENS - int(usage.prompt_tokens) - 24
        if retry_max > max_out:
            try:
                response = _create(retry_max)
            except BadRequestError:
                response = _create(max(256, retry_max - 512))
            choice = response.choices[0]

    if choice.finish_reason == "length":
        raise RuntimeError(
            "Judge still truncated JSON after using the full context window for completion."
        )

    content = choice.message.content
    if not content or not str(content).strip():
        raise ValueError("Judge model returned empty content")

    text = str(content).strip()
    if text.startswith("```"):
        lines = text.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip().startswith("```"):
            lines = lines[:-1]
        text = "\n".join(lines).strip()

    return CriterionEvaluation.model_validate_json(text)


# Run criterion-by-criterion evaluation on a subset (human-labeled rows; N products × 5 calls)
results_df_cb = pd.read_csv(OUTPUT_FILE_PATH)
subset_cb = results_df_cb[
    (results_df_cb["fluency"] != "") & (results_df_cb["fluency"].notna())
].reset_index(drop=True)

print(
    f"Isolated-criterion judge on {len(subset_cb)} products (5 API calls per product).\n"
)

atomic_rows = []
for _, row in subset_cb.iterrows():
    per_crit = {}
    for crit in QUALITY_CRITERIA:
        ev = judge_single_criterion(
            description=row["generated_description"],
            product_name=row["product_name"],
            attributes=row["Product_attribute_list"],
            material=row["material"],
            warranty=row["warranty"],
            criterion=crit,
        )
        per_crit[crit] = ev.verdict
    atomic_rows.append({"product_name": row["product_name"], **per_crit})

atomic_df = pd.DataFrame(atomic_rows)

print("Isolated verdicts (all subset rows):")
print(atomic_df.to_string(index=False))
print()

print("Agreement: isolated judge vs human (exact label per criterion)")
for crit in QUALITY_CRITERIA:
    agree = (
        atomic_df[crit].astype(str).values == subset_cb[crit].astype(str).values
    ).sum()
    print(f"  {crit}: {agree}/{len(subset_cb)}")

_bulk_cols = [f"judge_{c}" for c in QUALITY_CRITERIA]
if all(c in results_df_cb.columns for c in _bulk_cols):
    bulk_subset = subset_cb[_bulk_cols].rename(
        columns={f"judge_{c}": c for c in QUALITY_CRITERIA}
    )
    print("\nAgreement: isolated judge vs single-call judge (same CSV columns)")
    for crit in QUALITY_CRITERIA:
        agree = (
            atomic_df[crit].astype(str).values == bulk_subset[crit].astype(str).values
        ).sum()
        print(f"  {crit}: {agree}/{len(subset_cb)}")


Isolated-criterion judge on 10 products (5 API calls per product).

Isolated verdicts (all subset rows):
                   product_name fluency grammar tone length grounding
            Apple iPhone 15 Pro    good    good good     ok      good
       Samsung Galaxy S24 Ultra    good      ok good   good      good
             Google Pixel 8 Pro    good    good good     ok      good
     Sony WH-1000XM5 Headphones    good    good good     ok      good
Bose QuietComfort Ultra Earbuds      ok      ok good     ok       bad
      Amazon Echo Dot (5th Gen)    good    good good     ok      good
        Dell XPS 13 9310 Laptop    good    good good     ok      good
     Apple MacBook Air 13″ (M3)    good    good good     ok      good
       Microsoft Surface Pro 10    good    good good     ok      good
          Garmin Forerunner 255    good    good good     ok      good

Agreement: isolated judge vs human (exact label per criterion)
  fluency: 3/10
  grammar: 6/10
  tone: 4/10
  length: 1/10


**Criterion-by-Criterion Analysis:**

**Run summary (10 products, 5 API calls per product).** Isolated verdicts: iPhone 15 Pro / S24 Ultra / Pixel 8 Pro / Sony WH-1000XM5 / Bose QC Ultra / Echo Dot / Dell XPS 13 / MacBook Air M3 / Surface Pro 10 / Garmin Forerunner 255 — mostly **`good`** on fluency, grammar, tone, and grounding; **length** is mostly **`ok`** (Dell and Surface **`bad`**). **vs human:** fluency **4/10**, grammar **6/10**, tone **4/10**, length **0/10**, grounding **1/10**. **vs single-call judge (CSV columns):** fluency **6/10**, grammar **10/10**, tone **9/10**, length **4/10**, grounding **10/10**.

Compared to the **single-call** judge on the same rows (section 6.3 above), agreement between the two judge setups is **perfect for grammar and grounding (10/10)**, **very high for tone (9/10)**, **moderate for fluency (6/10)**, and **low for length (4/10)**. So isolation mostly reproduces the bulk judge except where **fluency** and especially **length** diverge.

1. **Did isolating criteria change the results?**
   - **Yes, selectively.** Against the stored single-call judge columns, **grammar** and **grounding** labels are identical for all 10 products; **tone** differs on only one row. **Fluency** disagrees on 4/10 rows (isolated is more often **`good`** in the printed table). **Length** disagrees on 6/10 rows: the isolated judge labels almost everything **`ok`** (with **`bad`** on Dell XPS 13 and Surface Pro 10), whereas the joint judge had aligned more often with human **`good`** on length in section 6.3. That pattern shows up in the isolated-vs-bulk length agreement (**4/10**).

2. **Why might this approach lead to different outcomes?**
   - **Narrow focus and rubric salience** — With only one criterion in the system prompt, the model may apply bands (especially the word-count rules for **length**) more mechanically, whereas a single multi-field JSON call can implicitly “smooth” labels across criteria.
   - **No cross-criterion context** — In joint evaluation, strong **fluency**/**tone** might subtly cohere with a **length** rating; in isolation, **length** is judged without that halo.
   - **Residual randomness** — Separate completions per criterion add independent sampling noise; that plausibly explains the **fluency** splits (6/10 match) even when **grammar**/**grounding** stay locked to the same labels.

3. **Did agreement with human scores improve?**
   - **Mostly no; length got much worse.** Using the same 10-row comparison as in section 6.3: **fluency** stays **4/10**, **grammar** **6/10**, **grounding** **1/10** — unchanged. **Tone** drops from **5/10** to **4/10**. **Length** collapses from **5/10** to **0/10**: every isolated length label mismatches the human, consistent with humans often rating length **`good`** while the isolated judge sticks to **`ok`**/**`bad`** from the strict word bands. So isolated judging did not improve human alignment overall; it **preserved** agreement where the bulk judge was already tied to humans (**fluency**, **grammar**, **grounding**) but **hurt** on **tone** and catastrophically on **length** for this sample.

### 6.5 Final Analysis and Reflection

#### Question 1: Trade-offs between human evaluation and LLM-as-a-judge

Consider: cost, scale, consistency, accuracy

Humans don't scale well and cost real time, but they’re still the best check for "would a shopper trust this?" Labels vary unless you train raters and refresh often.
Personally, it took me a long time to analyze each and every description and think what shoudl be the best rating per criterion. It is slow, costly, and subject to fatigue.

An LLM judge scales cheaply and behaves the same way run to run, which is handy for monitoring. Here it still disagreed with us often (tone, length, grounding), so it's a helper, not a replacement for human spot checks.

Except for minor disagreements with me about ok/good ratings - it most notably can't count words reliably. This reminds me the example of `How many r's are in 'strawberry'?`. If I can't rely on this simple task, it makes me suspicious of every output the LLM gives...

**Human Evaluation:**
- Pros:
  - Catches nuance and obvious mistakes rubrics miss; no model API needed.
- Cons:
  - Slow and costly at volume; agreement between humans isn't guaranteed on subjective tasks (like rating the `tone` rubric).

**LLM-as-a-Judge:**
- Pros:
  - Fast, cheap per item at scale; easy to wire into pipelines and get structured scores plus short reasons.
- Cons:
  - API cost and vendor lock-in; can be wrong with confidence and misaligned with humans without calibration.

#### Question 2: Recommendation for production system

For a production system generating thousands of descriptions daily:

Ship with **automated generation plus layered checks**, not human review on every line. Use **cheap deterministic gates first** (word count, banned phrases, required attributes present), then an **LLM judge** to flag likely issues on fluency, tone, and grounding. **Humans** should audit a **random sample** plus anything the judge marks `bad` or that's high stakes (regulated claims, big sellers).
Tasks like word-count with clear calculations - do programtically (not asking the LLM to count).

**Recommended approach:**
- Generate at scale; auto-filter obvious failures; LLM-judge the rest; human spot-check and handle escalations.

**Justification:**
- Full human QA on thousands a day doesn't scale; our judge tracks the rubric well enough for triage but still disagrees with people, so it shouldn't be the only gate.

**Implementation considerations:**
- Log scores, reasons, and model version; revisit prompts when the generator changes; cap judge cost (batching, smaller model for screening); define what happens when the judge and rules conflict (e.g. always send `bad` grounding to a person or a second pass).

---
## Summary and Submission

### Deliverables Checklist

- [V] Task 1: Rubric definitions and pass/fail formula
- [V] Task 2: Code for description generation
- [V] Task 3: `assignment_01.csv` with manual evaluations (10-15 products)
- [V] Task 4: Experiment documentation + code for successful experiments
- [V] Task 5: Judge model implementation with Pydantic schema
- [V] Task 6: Judge analysis, comparisons, and reflection

### Files to Submit

1. `assignment_01_solution.ipynb` (this notebook)
2. `assignment_01.csv` (with all evaluations)

**Due Date:** April 5, 2026